In [14]:
# Importe les outils de préparation des données et charge le jeu d'entraînement.
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
df=pd.read_csv("../data/raw/aug_train.csv")

In [4]:
# Définit les variables explicatives et la cible, en excluant l'identifiant du candidat.
X = df.drop(columns=["target", "enrollee_id"])
y = df["target"].astype(int)

In [5]:
# Crée des ensembles d'entraînement et de test en conservant la proportion des classes.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
# Convertit les valeurs textuelles extrêmes de l'expérience en nombres, puis vérifie le résultat.
def convertir_experience(s):
    return pd.to_numeric(s.replace({"<1": "0", ">20": "21"}))

exp = convertir_experience(X_train["experience"])
print(exp.dtype, exp.isna().sum(), exp.min(), exp.max())

float64 50 0.0 21.0


In [7]:
# Convertit les modalités du délai depuis le dernier emploi en nombres, puis vérifie le résultat.
def convertir_last_new_job(s):
    return pd.to_numeric(s.replace({"never": "0", ">4": "5"}))

lnj = convertir_last_new_job(X_train["last_new_job"])
print(lnj.dtype, lnj.isna().sum(), lnj.min(), lnj.max())

float64 343 0.0 5.0


In [8]:
# Examine le nombre de valeurs manquantes et le type de chaque variable de l'ensemble d'entraînement.
print(X_train.isna().sum())
print(X_train.dtypes)


city                         0
city_development_index       0
gender                    3574
relevent_experience          0
enrolled_university        317
education_level            362
major_discipline          2258
experience                  50
company_size              4714
company_type              4881
last_new_job               343
training_hours               0
dtype: int64
city                          str
city_development_index    float64
gender                        str
relevent_experience           str
enrolled_university           str
education_level               str
major_discipline              str
experience                    str
company_size                  str
company_type                  str
last_new_job                  str
training_hours              int64
dtype: object


In [9]:
# Convertit et impute les variables numériques avec la médiane calculée sur l'ensemble d'entraînement.
def convertir_numeriques(df_):
    df_ = df_.copy()
    df_["experience"] = convertir_experience(df_["experience"])
    df_["last_new_job"] = convertir_last_new_job(df_["last_new_job"])
    return df_
branche_num = Pipeline([
    ("conversion", FunctionTransformer(convertir_numeriques)),
    ("imputation", SimpleImputer(strategy="median")),
])

cols_g1 = ["experience", "last_new_job"]

Xtr = branche_num.fit_transform(X_train[cols_g1])
Xte = branche_num.transform(X_test[cols_g1])

print(Xtr.shape, Xte.shape)
print(np.isnan(Xtr).sum(), np.isnan(Xte).sum())
print(branche_num.named_steps["imputation"].statistics_)



(15326, 2) (3832, 2)
0 0
[9. 1.]


In [10]:
# Remplace les catégories manquantes par « Inconnu », puis les encode en variables indicatrices.
cols_inconnu = ["company_size", "company_type", "gender", "education_level"]

branche_inconnu = Pipeline([
    ("imputation", SimpleImputer(strategy="constant", fill_value="Inconnu")),
    ("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

Xtr = branche_inconnu.fit_transform(X_train[cols_inconnu])
Xte = branche_inconnu.transform(X_test[cols_inconnu])

print(Xtr.shape, Xte.shape)
print(branche_inconnu.named_steps["encodage"].categories_)

(15326, 26) (3832, 26)
[array(['10/49', '100-500', '1000-4999', '10000+', '50-99', '500-999',
       '5000-9999', '<10', 'Inconnu'], dtype=object), array(['Early Stage Startup', 'Funded Startup', 'Inconnu', 'NGO', 'Other',
       'Public Sector', 'Pvt Ltd'], dtype=object), array(['Female', 'Inconnu', 'Male', 'Other'], dtype=object), array(['Graduate', 'High School', 'Inconnu', 'Masters', 'Phd',
       'Primary School'], dtype=object)]


In [11]:
branche_na = Pipeline([
    ("imputation", SimpleImputer(strategy="constant", fill_value="Non applicable")),
    ("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
branche_freq = Pipeline([
    ("imputation", SimpleImputer(strategy="most_frequent")),
    ("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

Xtr_na = branche_na.fit_transform(X_train[["major_discipline"]])
Xtr_freq = branche_freq.fit_transform(X_train[["enrolled_university"]])

print(Xtr_na.shape, Xtr_freq.shape)
print(branche_na.named_steps["encodage"].categories_)
print(branche_freq.named_steps["imputation"].statistics_)
print(branche_freq.named_steps["encodage"].categories_)

(15326, 7) (15326, 3)
[array(['Arts', 'Business Degree', 'Humanities', 'No Major',
       'Non applicable', 'Other', 'STEM'], dtype=object)]
['no_enrollment']
[array(['Full time course', 'Part time course', 'no_enrollment'],
      dtype=object)]


In [12]:
branche_echelle = Pipeline([("echelle", StandardScaler())])
branche_bin = Pipeline([("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
branche_ville = Pipeline([("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

Xtr_e = branche_echelle.fit_transform(X_train[["city_development_index", "training_hours"]])
Xtr_b = branche_bin.fit_transform(X_train[["relevent_experience"]])
Xtr_v = branche_ville.fit_transform(X_train[["city"]])

print(Xtr_e.shape, Xtr_b.shape, Xtr_v.shape)
print(Xtr_e.mean(axis=0).round(3), Xtr_e.std(axis=0).round(3))

(15326, 2) (15326, 2) (15326, 122)
[-0. -0.] [1. 1.]


In [13]:
branche_echelle = Pipeline([("echelle", StandardScaler())])
branche_bin = Pipeline([("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
branche_ville = Pipeline([("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

Xtr_e = branche_echelle.fit_transform(X_train[["city_development_index", "training_hours"]])
Xtr_b = branche_bin.fit_transform(X_train[["relevent_experience"]])
Xtr_v = branche_ville.fit_transform(X_train[["city"]])

print(Xtr_e.shape, Xtr_b.shape, Xtr_v.shape)
print(Xtr_e.mean(axis=0).round(3), Xtr_e.std(axis=0).round(3))

(15326, 2) (15326, 2) (15326, 122)
[-0. -0.] [1. 1.]


In [18]:
branche_num = Pipeline([
    ("conversion", FunctionTransformer(convertir_numeriques, feature_names_out="one-to-one")),
    ("imputation", SimpleImputer(strategy="median")),
])

preprocess = ColumnTransformer([
    ("num_mediane", branche_num, ["experience", "last_new_job"]),
    ("cat_inconnu", branche_inconnu, ["company_size", "company_type", "gender", "education_level"]),
    ("cat_na", branche_na, ["major_discipline"]),
    ("cat_freq", branche_freq, ["enrolled_university"]),
    ("echelle", branche_echelle, ["city_development_index", "training_hours"]),
    ("binaire", branche_bin, ["relevent_experience"]),
    ("ville", branche_ville, ["city"]),
])

Xtr_prep = preprocess.fit_transform(X_train)
Xte_prep = preprocess.transform(X_test)

print(Xtr_prep.shape, Xte_prep.shape)      # (15326, 164) (3832, 164)
noms = preprocess.get_feature_names_out()
print(len(noms), noms[:5])

(15326, 164) (3832, 164)
164 ['num_mediane__experience' 'num_mediane__last_new_job'
 'cat_inconnu__company_size_10/49' 'cat_inconnu__company_size_100-500'
 'cat_inconnu__company_size_1000-4999']


In [19]:
# 1. Le pipeline n'a pas modifié les données d'origine
print(X_train.isna().sum().sum())

# 2. Les médianes viennent bien du train
print(preprocess.named_transformers_["num_mediane"].named_steps["imputation"].statistics_)

# 3. Les noms des 164 colonnes
noms = preprocess.get_feature_names_out()
print(len(noms), noms[:5])

# 4. La ville qui n'existe que dans le test
villes_test = set(X_test["city"]) - set(X_train["city"])
print(villes_test)
masque = X_test["city"].isin(villes_test).to_numpy()
print(masque.sum(), Xte_prep[masque][:, -122:].sum())

16499
[9. 1.]
164 ['num_mediane__experience' 'num_mediane__last_new_job'
 'cat_inconnu__company_size_10/49' 'cat_inconnu__company_size_100-500'
 'cat_inconnu__company_size_1000-4999']
{'city_140'}
1 0.0
